# Fine-Tune SmolLM2-135M on Industrial Telemetry Dataset
This notebook fine-tunes `HuggingFaceTB/SmolLM2-135M-Instruct` on our custom dataset of 3,156 samples for fast, reliable offline edge inference.

In [ ]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load train and validation splits
dataset = load_dataset("json", data_files={
    "train": "data/train_chatml.jsonl",
    "validation": "data/val_chatml.jsonl"
})
print(dataset)

In [ ]:
training_args = TrainingArguments(
    output_dir="./smollm2-135m-industrial",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
    max_seq_length=512
)

trainer.train()
trainer.save_model("./smollm2-135m-industrial-adapter")
tokenizer.save_pretrained("./smollm2-135m-industrial-adapter")